# [1장 3강 심화] - 학습 파이프라인 순서 이해 실습

## 실습 목표
- 데이터 → 모델 출력 → 손실 → gradient 초기화 → 역전파 → 업데이트 → 검증이라는 큰 흐름과 단계별 산출물을 구분해야한다.
- 특히 model.eval()은 평가 모드를 정하고, torch.no_grad()는 gradient 추적을 끄며, 둘 다 파라미터 업데이트 명령은 아니라는 점을 계약으로 남겨야 한다.
- 단계 사이의 의존 관계를 근거로 순서 오류와 검증 오염을 찾아낸다.

실행 기록 수집 → 단계 의존성 검사 → 한 batch 학습 코드를 올바른 순서로 복구 → 검증 오염 판정

## 상황 자료

```
02:10 validation start
02:10 valid_loss=0.412
02:10 optimizer.step called
02:10 validation end
```

### 문제 1. 뒤섞인 실행 기록 진단하기
확인해야 할 규칙
- backward는 loss 뒤에 오는가 (오차를 구해야 gradient를 계산할 수 있으므로)
- step은 backward 뒤에 오는가 (gradient를 계산해야 가중치를 갱신할 수 있으므로)
- zero_grad는 새 gradient 계산(backward) 전에 오는가 (이전 gradient를 지워야 하므로)

제출할 결과
- 첫 위반 지점: 규칙을 처음 어긴 곳
- 올바른 순서: 복구한 정상 순서
- 이유: 왜 그 순서여야 하는지

In [3]:
observed = ["forward", "loss", "step", "zero_grad", "backward"]

1. 첫 위반 지점: step이 backward 보다 먼저 앞서 있음
2. 올바른 순서: zero_grad -> forward -> loss -> backward -> step
3. 이유: 전체 흐름은 아래와 같으며, 각 단계는 앞 단계의 산출물에 의존한다.
    - Dataset -> DataLoader -> X_batch, y_batch
     - -> zero_grad()        : 이전 gradient 청소
     - -> model(X_batch)     : forward, 예측값(prediction) 생성
     - -> loss_fn(pred, y)   : 예측과 정답을 비교해 오차(loss) 계산
     - -> backward()         : loss를 미분해 gradient 계산
     - -> optimizer.step()   : gradient로 parameter 업데이트
     - -> 다음 batch

### 문제 2. batch 학습 코드를 직접 복구하기
- batch 하나를 학습시켰을 때 파라미터가 실제로 갱신되도록 코드를 완성할 것
- 이를 loss의 변화가 아니라 weight의 전후 변화로 검증할 것

- seed를 고정하고 모델, loss 함수, optimizer를 생성한다.
- 학습 계약 순서(순전파 → loss 계산 → 역전파 → optimizer step)대로 한 step을 실행한다.
- 업데이트 직전의 weight를 복사해 두고, step 이후의 weight와 비교한다.
- loss 값과 weight 변경 여부(weight_changed)를 출력한다.

In [ ]:
import torch
from torch import nn

SEED = 42
# Sets the seed for generating random numbers on all devices. Returns a torch.Generator object.
torch.manual_seed(SEED)

# (function) def tensor(
#     data: Any,
#     dtype: dtype | None = None,
#     device: DeviceLikeType | None = None,
#     requires_grad: bool = False,
#     pin_memory: bool = False
# ) -> Tensor

# 인자 1개: 리스트 [[1.0],[2.0]] 하나를 통째로 넘김
# tensor = (임의 차원의 숫자 배열) + (연산 기록으로 미분해주는 autograd) + (GPU에서 굴러가는 능력)
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])
# print(x.type(), y.type()) # torch.FloatTensor

model = nn.Linear(1, 1) # (입력 특징 수, 출력 특징 수)
# Linear: 일차함수(선형 변환), y = wx + b
# nn.Linear(1, 1)은 이 W와 b를 각각 자동으로 하나씩 만들어서 내부에 넣어둔다
# print(model) # Linear(in_features=1, out_features=1, bias=True)
# print(model.weight) # tensor([[0.7645]], requires_grad=True)
# print(model.bias) # tensor([0.8300], requires_grad=True)
# 층을 만드는 순간 학습의 출발점으로 weight·bias가 채워지고, 그 값은 1/√(입력 수) 범위의 규칙 있는 랜덤 값

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) # Stochastic Gradient Descent
# model.parameters() 인자는 optimizer에게 갱신할 대상(W, b)이 뭔지 명단을 등록해주는 것

# print(list(model.parameters()))
# tensor([[0.7645]], requires_grad=True)
# tensor([0.8300], requires_grad=True)]

# .detach() — torch.Tensor의 메서드 (PyTorch 전용)
#   - autograd 그래프에서 분리된 새 텐서를 반환 (원본은 그대로 둠)
#   - 반환된 텐서는 requires_grad=False → 이후 연산이 gradient에 영향 없음
#   - 데이터(값)는 원본과 메모리를 공유 → 값 복사가 아님. 독립 사본이 필요하면 .clone()을 함께 사용
before = model.weight.detach().clone()
print(f"before optimizer.step weight: {before}") # tensor([[0.7645]]) # torch.FloatTensor

optimizer.zero_grad()       # 이전 gradient 청소
pred = model(x)             # forward, 예측값(prediction) 생성
loss = loss_fn(pred, y)     # 예측과 정답을 비교해 오차(loss) 계산
loss.backward()             # loss를 미분해 gradient 계산
optimizer.step()            # gradient로 parameter 업데이트

after = model.weight.detach().clone()
print(f"after optimzer.step weight: {after}")

before optimizer.step weight: tensor([[0.7645]])
after optimzer.step weight: tensor([[1.1333]])


### 문제 3. validation 수치를 믿어도 되는지 판단하기
- 각 run의 시작·종료 checksum을 비교해, validation 중 weight가 바뀌었는지 확인한다.
- weight가 바뀐 run은 valid loss와 무관하게 오염된 것으로 보고 제외한다.
- 신뢰 가능한 run 중 valid loss가 가장 작은 것을 고른다.
- eval()과 no_grad()만으로는 optimizer.step() 호출을 상쇄할 수 없는 이유를 서술한다.

상황 로그
- run A: before=18.20, after=18.20, valid_loss=0.44
- run B: before=18.20, after=18.31, valid_loss=0.39

제출물
- trusted_runs (신뢰 가능한 run 목록)
- selected (최종 선택된 run)
- run B를 제외한 근거

In [19]:
runs_results = [
    {"name": "A", "before": 18.20, "after": 18.20, "valid_loss": 0.44},
    {"name": "B", "before": 18.20, "after": 18.31, "valid_loss": 0.39},
]

trusted_runs = [r for r in runs_results if r["before"] == r["after"]]
selected = min(trusted_runs, key=lambda r: r["valid_loss"])["name"] if trusted_runs else "보류"
print(f"trsuted_runs: {trusted_runs}")
print(f"selected: {selected}")

trsuted_runs: [{'name': 'A', 'before': 18.2, 'after': 18.2, 'valid_loss': 0.44}]
selected: A


- trsuted_runs: [{'name': 'A', 'before': 18.2, 'after': 18.2, 'valid_loss': 0.44}]
- selected: A
- run B를 제외한 근거: 검증 데이터를 통해 학습이 일어났기 때문에 공정한 검증 수치가 아니다.